# CIFAR-10 Project Runner

Starter notebook for the CIFAR-10 image classification project. The notebook provides dataset loading, device setup, transforms, dataloaders, training curves, and evaluation metrics. You fill in model architecture, loss function, optimizer, and any preprocessing choices you want to try.

## Initialize Project

- Import project dependencies
- Seed PyTorch for repeatable runs
- Detect the best available device: GPU, Apple Silicon, or CPU
- Use `torchvision` to load CIFAR-10

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

torch.manual_seed(42)


def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


device = get_device()
device

## Locate Dataset

- Store CIFAR-10 files under `sandbox/cifar10/data`
- Let `torchvision.datasets.CIFAR10` download the dataset automatically
- No separate `setup.py` file is needed for this project

In [ ]:
CURRENT_DIR = Path.cwd()
CIFAR_DIR = CURRENT_DIR if CURRENT_DIR.name == "cifar10" else CURRENT_DIR / "sandbox" / "cifar10"
DATA_DIR = CIFAR_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR

## Choose Preprocessing

- CIFAR-10 images are color images with shape `(3, 32, 32)` after `ToTensor()`
- Transforms are where you can normalize images or add data augmentation
- Start simple, then change one preprocessing choice at a time

In [ ]:
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2470, 0.2435, 0.2616)

# TODO: Change these transforms if you want to experiment with preprocessing.
# Examples to consider:
# transforms.RandomHorizontalFlip()
# transforms.RandomCrop(32, padding=4)
# transforms.ColorJitter(...)
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

## Load CIFAR-10

- Download train and test splits with `torchvision`
- Use smaller subsets by default so early experiments run quickly
- Increase the subset sizes after your model runs correctly

In [ ]:
train_data = datasets.CIFAR10(root=str(DATA_DIR), train=True, download=True, transform=train_transform)
test_data = datasets.CIFAR10(root=str(DATA_DIR), train=False, download=True, transform=test_transform)

class_names = train_data.classes
class_names

## Build DataLoaders

- Wrap CIFAR-10 examples in `DataLoader`s
- Shuffle the training loader
- Keep the test loader deterministic

In [ ]:
train_subset_size = 10000
test_subset_size = 2000
batch_size = 64

train_subset = Subset(train_data, range(train_subset_size))
test_subset = Subset(test_data, range(test_subset_size))

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=128)

len(train_subset), len(test_subset)

## Visualize Dataset

- Display a few CIFAR-10 images
- Use this to sanity-check transforms and labels before training

In [ ]:
def show_image(tensor_image, ax, title):
    mean = torch.tensor(cifar10_mean).view(3, 1, 1)
    std = torch.tensor(cifar10_std).view(3, 1, 1)
    image = tensor_image.cpu() * std + mean
    image = image.clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")


fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, index in zip(axes.ravel(), range(10)):
    image, label = train_data[index]
    show_image(image, ax, class_names[label])

plt.tight_layout()

## Define Model

- You define the model here
- Input shape is `(batch, channels, height, width)`
- CIFAR-10 images have shape `(batch, 3, 32, 32)`
- Output should have shape `(batch, 10)` for the 10 CIFAR-10 classes

In [ ]:
class CIFAR10Model(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: Define model layers here.
        # Example layer types to consider:
        # self.conv = nn.Conv2d(...)
        # self.pool = nn.MaxPool2d(...)
        # self.classifier = nn.Linear(...)
        raise NotImplementedError("Define your model layers in __init__")

    def forward(self, x):
        # TODO: Define the forward pass.
        raise NotImplementedError("Define your model forward pass")


model = CIFAR10Model().to(device)
model

## Choose Loss and Optimizer

- Choose a loss function for 10-class classification
- Choose an optimizer and learning rate
- For CIFAR-10 labels, `nn.CrossEntropyLoss()` is a reasonable first choice

In [ ]:
# TODO: Choose a loss function.
# Example: loss_fn = nn.CrossEntropyLoss()
loss_fn = None

# TODO: Choose an optimizer.
# Example: optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer = None

if loss_fn is None:
    raise NotImplementedError("Choose a loss function before training")
if optimizer is None:
    raise NotImplementedError("Choose an optimizer before training")

## Train Model

- Train the model on CIFAR-10 images
- Move each batch to the selected device
- Track train and test loss/accuracy across epochs

In [ ]:
def evaluate_loss_accuracy(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            logits = model(batch_X)
            loss = loss_fn(logits, batch_y)
            total_loss += loss.item()
            correct += (logits.argmax(dim=1) == batch_y).sum().item()
            total += batch_y.size(0)

    return total_loss / len(loader), correct / total


history = {"train_loss": [], "train_accuracy": [], "test_loss": [], "test_accuracy": []}

for epoch in range(5):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        logits = model(batch_X)
        loss = loss_fn(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(dim=1) == batch_y).sum().item()
        total += batch_y.size(0)

    train_loss = total_loss / len(train_loader)
    train_accuracy = correct / total
    test_loss, test_accuracy = evaluate_loss_accuracy(model, test_loader, loss_fn)

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["test_loss"].append(test_loss)
    history["test_accuracy"].append(test_accuracy)

    print(
        f"epoch {epoch + 1}: "
        f"train_loss={train_loss:.4f}, train_accuracy={train_accuracy:.3f}, "
        f"test_loss={test_loss:.4f}, test_accuracy={test_accuracy:.3f}"
    )

## Plot Training Curves

- Plot train/test loss across epochs
- Plot train/test accuracy across epochs
- Use the curves to judge whether the model is learning, overfitting, or underfitting

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(epochs, history["train_loss"], label="train loss")
axes[0].plot(epochs, history["test_loss"], label="test loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cross entropy")
axes[0].legend()

axes[1].plot(epochs, history["train_accuracy"], label="train accuracy")
axes[1].plot(epochs, history["test_accuracy"], label="test accuracy")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].set_ylim(0, 1)
axes[1].legend()

plt.tight_layout()

## Evaluate Model

- Run the trained model on held-out CIFAR-10 images
- Report accuracy, class-level metrics, and a confusion matrix
- Use these metrics to understand which classes are easy or hard for your model

In [ ]:
def collect_predictions(model, loader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(device)
            logits = model(batch_X)
            predictions = logits.argmax(dim=1).cpu()
            y_true.extend(batch_y.tolist())
            y_pred.extend(predictions.tolist())

    return np.asarray(y_true), np.asarray(y_pred)


y_true, y_pred = collect_predictions(model, test_loader)
cm = confusion_matrix(y_true, y_pred)

print(f"accuracy: {accuracy_score(y_true, y_pred):.3f}")
print("\nclassification report:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(class_names)), class_names, rotation=45, ha="right")
ax.set_yticks(range(len(class_names)), class_names)
ax.set_xlabel("predicted")
ax.set_ylabel("true")
ax.set_title("Confusion Matrix")

for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        ax.text(col, row, cm[row, col], ha="center", va="center", fontsize=8)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()